# ViT Unfreeze Comparison (2 / 4 / 6)

This notebook keeps the dataset split, augmentation, and optimization setup fixed, and compares stage-2 fine-tuning with the last 2, 4, or 6 ViT transformer blocks unfrozen.


In [2]:
# Local/WSL workflow: no Colab drive mount needed.


In [3]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import mixed_precision

gpus = tf.config.list_physical_devices("GPU")
for gpu in gpus:
    tf.config.experimental.set_memory_growth(gpu, True)

if gpus:
    mixed_precision.set_global_policy("mixed_float16")

print("TF version:", tf.__version__)
print("GPU:", gpus)
print("Mixed precision policy:", mixed_precision.global_policy())

import matplotlib.pyplot as plt
import numpy as np
import os
import random
import keras_hub
from collections import Counter
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    roc_auc_score,
    roc_curve,
)


TF version: 2.20.0
GPU: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
Mixed precision policy: <DTypePolicy "mixed_float16">


In [4]:
def count_files_by_class(root_dir):
    counts = {}
    for class_name in sorted(os.listdir(root_dir)):
        class_dir = os.path.join(root_dir, class_name)
        if not os.path.isdir(class_dir):
            continue
        counts[class_name] = sum(
            1
            for file_name in os.listdir(class_dir)
            if file_name.lower().endswith((".jpg"))
        )
    return counts


def summarize_labels(ds, class_names):
    counts = Counter()
    for _, labels in ds:
        labels = labels.numpy().astype("int32").reshape(-1)
        counts.update(labels.tolist())
    return {class_names[idx]: counts.get(idx, 0) for idx in range(len(class_names))}


def collect_predictions(ds, model):
    y_true = []
    y_prob = []
    for xb, yb in ds:
        pb = model.predict(xb, verbose=0).reshape(-1)
        y_true.append(yb.numpy().reshape(-1))
        y_prob.append(pb)
    y_true = np.concatenate(y_true).astype("int32")
    y_prob = np.concatenate(y_prob)
    return y_true, y_prob


In [5]:
data_dir = "/mnt/e/DDSM/ROI"
assert os.path.exists(data_dir), f"Dataset path not found: {data_dir}"
raw_class_counts = count_files_by_class(data_dir)
print("Raw file counts by class:", raw_class_counts)


Raw file counts by class: {'ROI_Benign': 1402, 'ROI_Cancer': 1428}


In [6]:
img_size = (224, 224)
batch_size = 16
seed = 123
AUTOTUNE = tf.data.AUTOTUNE

os.environ["PYTHONHASHSEED"] = str(seed)
random.seed(seed)
np.random.seed(seed)
tf.random.set_seed(seed)
keras.utils.set_random_seed(seed)
try:
    tf.config.experimental.enable_op_determinism()
except Exception:
    pass

train_ds = tf.keras.utils.image_dataset_from_directory(
    data_dir,
    labels="inferred",
    label_mode="binary",
    validation_split=0.2,
    subset="training",
    seed=seed,
    image_size=img_size,
    batch_size=batch_size,
    shuffle=True,
)

temp_ds = tf.keras.utils.image_dataset_from_directory(
    data_dir,
    labels="inferred",
    label_mode="binary",
    validation_split=0.2,
    subset="validation",
    seed=seed,
    image_size=img_size,
    batch_size=batch_size,
    shuffle=True,
)

class_names = train_ds.class_names
print("Classes:", class_names)
print(f"Positive class for AUC/ROC: {class_names[1]}")

temp_card = tf.data.experimental.cardinality(temp_ds).numpy()
if temp_card < 2:
    raise ValueError(f"Validation split is too small: only {temp_card} batch(es). Reduce batch_size or use more data.")

val_batches = max(1, temp_card // 2)
test_batches = temp_card - val_batches
if test_batches == 0:
    raise ValueError("Test split is empty. Reduce batch_size or use more data.")

val_ds = temp_ds.take(val_batches)
test_ds = temp_ds.skip(val_batches)

def prepare(ds, shuffle=False):
    if shuffle:
        ds = ds.shuffle(1000, seed=seed, reshuffle_each_iteration=True)
    return ds.prefetch(AUTOTUNE)

train_ds_prep = prepare(train_ds, shuffle=True)
val_ds_prep = prepare(val_ds)
test_ds_prep = prepare(test_ds)

print("Train batches:", tf.data.experimental.cardinality(train_ds).numpy())
print("Validation batches:", tf.data.experimental.cardinality(val_ds).numpy())
print("Test batches:", tf.data.experimental.cardinality(test_ds).numpy())

train_label_counts = summarize_labels(train_ds, class_names)
val_label_counts = summarize_labels(val_ds, class_names)
test_label_counts = summarize_labels(test_ds, class_names)

print("Train label counts:", train_label_counts)
print("Validation label counts:", val_label_counts)
print("Test label counts:", test_label_counts)


Found 2830 files belonging to 2 classes.
Using 2264 files for training.


I0000 00:00:1773202225.189399   19847 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 3584 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 3060 Laptop GPU, pci bus id: 0000:01:00.0, compute capability: 8.6


Found 2830 files belonging to 2 classes.
Using 566 files for validation.
Classes: ['ROI_Benign', 'ROI_Cancer']
Positive class for AUC/ROC: ROI_Cancer
Train batches: 142
Validation batches: 18
Test batches: 18


2026-03-11 12:11:05.073721: I tensorflow/core/framework/local_rendezvous.cc:407] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence
2026-03-11 12:11:09.808538: I tensorflow/core/framework/local_rendezvous.cc:407] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence
2026-03-11 12:11:10.607494: W tensorflow/core/kernels/data/prefetch_autotuner.cc:55] Prefetch autotuner tried to allocate 18937856 bytes after encountering the first element of size 18937856 bytes.This already causes the autotune ram budget to be exceeded. To stay within the ram budget, either increase the ram budget or reduce element size


Train label counts: {'ROI_Benign': 1123, 'ROI_Cancer': 1141}
Validation label counts: {'ROI_Benign': 146, 'ROI_Cancer': 142}
Test label counts: {'ROI_Benign': 143, 'ROI_Cancer': 135}


In [7]:
for images, labels in train_ds.take(1):
    print("image batch:", images.shape)
    print("label batch:", labels.shape)
    print("labels:", labels[:8].numpy().reshape(-1))


image batch: (16, 224, 224, 3)
label batch: (16, 1)
labels: [0. 0. 0. 0. 1. 1. 0. 1.]


2026-03-11 12:11:13.630344: I tensorflow/core/framework/local_rendezvous.cc:407] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


In [8]:
data_augmentation = keras.Sequential(
    [
        keras.layers.RandomFlip("horizontal"),
        keras.layers.RandomRotation(0.10),
        keras.layers.RandomZoom(0.20),
        keras.layers.RandomTranslation(0.05, 0.05),
        keras.layers.RandomContrast(0.20),
    ],
    name="data_augmentation",
)


In [9]:
def build_vit_model():
    backbone = keras_hub.models.ViTBackbone.from_preset(
        "vit_base_patch16_224_imagenet21k"
    )

    preprocessing = keras.Sequential(
        [keras.layers.Rescaling(1.0 / 255)],
        name="preprocessing",
    )

    inputs = keras.Input(shape=img_size + (3,))
    x = preprocessing(inputs)
    x = data_augmentation(x)
    x = backbone(x)
    x = x[:, 0, :]
    x = keras.layers.Dense(256, activation="relu")(x)
    x = keras.layers.BatchNormalization()(x)
    x = keras.layers.Dropout(0.5)(x)
    outputs = keras.layers.Dense(1, activation="sigmoid", dtype="float32", name="cancer_prob")(x)

    model = keras.Model(inputs, outputs)
    return model, backbone


def compile_binary_model(model, learning_rate, weight_decay=None):
    if weight_decay is None:
        optimizer = keras.optimizers.Adam(learning_rate=learning_rate)
    else:
        optimizer = keras.optimizers.AdamW(
            learning_rate=learning_rate,
            weight_decay=weight_decay,
        )

    model.compile(
        optimizer=optimizer,
        loss=keras.losses.BinaryCrossentropy(),
        metrics=[
            keras.metrics.AUC(name="auc"),
            keras.metrics.BinaryAccuracy(name="acc"),
            keras.metrics.Precision(name="precision"),
            keras.metrics.Recall(name="recall"),
        ],
    )


def find_transformer_blocks(backbone):
    all_backbone_layers = list(backbone._flatten_layers(include_self=False, recursive=True))
    transformer_blocks = [
        layer for layer in all_backbone_layers
        if "tranformer_block" in layer.name or "transformer_block" in layer.name
    ]
    if not transformer_blocks:
        candidate_names = [
            layer.name for layer in all_backbone_layers
            if any(token in layer.name.lower() for token in ["vit", "patch", "encoder", "block"])
        ]
        raise ValueError(
            "No transformer blocks found in backbone. Candidates: " + ", ".join(candidate_names[:30])
        )
    return all_backbone_layers, transformer_blocks


In [10]:
stage1_model, stage1_backbone = build_vit_model()
stage1_backbone.trainable = False
compile_binary_model(stage1_model, learning_rate=1e-4)

print("Backbone type:", type(stage1_backbone).__name__)
print("Model output shape:", stage1_model.output_shape)
print("Stage 1 trainable weights:", len(stage1_model.trainable_weights))

stage1_ckpt = "/mnt/e/SW_training_outputs/checkpoints/best_vit_stage1_comparison.keras"
cb1 = [
    keras.callbacks.ModelCheckpoint(
        stage1_ckpt,
        monitor="val_auc",
        mode="max",
        save_best_only=True,
        verbose=1,
    ),
    keras.callbacks.EarlyStopping(
        monitor="val_auc",
        mode="max",
        patience=3,
        restore_best_weights=True,
        verbose=1,
    ),
]

history1 = stage1_model.fit(
    train_ds_prep,
    validation_data=val_ds_prep,
    epochs=10,
    callbacks=cb1,
    verbose=1,
)


Backbone type: ViTBackbone
Model output shape: (None, 1)
Stage 1 trainable weights: 6
Epoch 1/10


2026-03-11 12:12:10.573122: E tensorflow/core/util/util.cc:131] oneDNN supports DT_HALF only on platforms with AVX-512. Falling back to the default Eigen-based implementation if present.
2026-03-11 12:18:13.756701: I tensorflow/core/kernels/data/shuffle_dataset_op.cc:453] ShuffleDatasetV3:4: Filling up shuffle buffer (this may take a while): 1 of 128
2026-03-11 12:18:14.152949: I tensorflow/core/kernels/data/shuffle_dataset_op.cc:483] Shuffle buffer filled.
2026-03-11 12:18:25.978257: I tensorflow/core/kernels/data/shuffle_dataset_op.cc:453] ShuffleDatasetV3:18: Filling up shuffle buffer (this may take a while): 10 of 1000
2026-03-11 12:18:39.110668: I tensorflow/core/kernels/data/shuffle_dataset_op.cc:453] ShuffleDatasetV3:18: Filling up shuffle buffer (this may take a while): 127 of 1000
2026-03-11 12:18:43.889282: I tensorflow/core/kernels/data/shuffle_dataset_op.cc:483] Shuffle buffer filled.
2026-03-11 12:18:43.995195: W tensorflow/core/kernels/data/prefetch_autotuner.cc:55] Prefe

142/142 ━━━━━━━━━━━━━━━━━━━━ 0s 116ms/step - acc: 0.5444 - auc: 0.5611 - loss: 0.8735 - precision: 0.5363 - recall: 0.5583
Epoch 1: val_auc improved from None to 0.66310, saving model to /mnt/e/SW_training_outputs/checkpoints/best_vit_stage1_comparison.keras

Epoch 1: finished saving model to /mnt/e/SW_training_outputs/checkpoints/best_vit_stage1_comparison.keras
142/142 ━━━━━━━━━━━━━━━━━━━━ 492s 223ms/step - acc: 0.5583 - auc: 0.5756 - loss: 0.8528 - precision: 0.5619 - recall: 0.5609 - val_acc: 0.6007 - val_auc: 0.6631 - val_loss: 0.6624 - val_precision: 0.6496 - val_recall: 0.5705
Epoch 2/10


2026-03-11 12:20:05.760417: I tensorflow/core/kernels/data/shuffle_dataset_op.cc:453] ShuffleDatasetV3:18: Filling up shuffle buffer (this may take a while): 53 of 1000
2026-03-11 12:20:15.852747: I tensorflow/core/kernels/data/shuffle_dataset_op.cc:453] ShuffleDatasetV3:18: Filling up shuffle buffer (this may take a while): 101 of 1000
2026-03-11 12:20:23.264696: I tensorflow/core/kernels/data/shuffle_dataset_op.cc:483] Shuffle buffer filled.


142/142 ━━━━━━━━━━━━━━━━━━━━ 0s 111ms/step - acc: 0.5669 - auc: 0.5891 - loss: 0.8301 - precision: 0.5812 - recall: 0.5715
Epoch 2: val_auc improved from 0.66310 to 0.70318, saving model to /mnt/e/SW_training_outputs/checkpoints/best_vit_stage1_comparison.keras

Epoch 2: finished saving model to /mnt/e/SW_training_outputs/checkpoints/best_vit_stage1_comparison.keras
142/142 ━━━━━━━━━━━━━━━━━━━━ 70s 301ms/step - acc: 0.5844 - auc: 0.6115 - loss: 0.8019 - precision: 0.5877 - recall: 0.5872 - val_acc: 0.6354 - val_auc: 0.7032 - val_loss: 0.6321 - val_precision: 0.6149 - val_recall: 0.7379
Epoch 3/10


2026-03-11 12:21:16.253925: I tensorflow/core/kernels/data/shuffle_dataset_op.cc:453] ShuffleDatasetV3:18: Filling up shuffle buffer (this may take a while): 55 of 1000
2026-03-11 12:21:32.538586: I tensorflow/core/kernels/data/shuffle_dataset_op.cc:483] Shuffle buffer filled.


142/142 ━━━━━━━━━━━━━━━━━━━━ 0s 110ms/step - acc: 0.6024 - auc: 0.6316 - loss: 0.7595 - precision: 0.6006 - recall: 0.6096
Epoch 3: val_auc improved from 0.70318 to 0.73485, saving model to /mnt/e/SW_training_outputs/checkpoints/best_vit_stage1_comparison.keras

Epoch 3: finished saving model to /mnt/e/SW_training_outputs/checkpoints/best_vit_stage1_comparison.keras
142/142 ━━━━━━━━━━━━━━━━━━━━ 56s 208ms/step - acc: 0.5989 - auc: 0.6306 - loss: 0.7551 - precision: 0.6030 - recall: 0.5977 - val_acc: 0.6528 - val_auc: 0.7349 - val_loss: 0.6006 - val_precision: 0.6738 - val_recall: 0.6376
Epoch 4/10
142/142 ━━━━━━━━━━━━━━━━━━━━ 0s 117ms/step - acc: 0.6080 - auc: 0.6562 - loss: 0.7248 - precision: 0.6133 - recall: 0.6123
Epoch 4: val_auc did not improve from 0.73485
142/142 ━━━━━━━━━━━━━━━━━━━━ 20s 133ms/step - acc: 0.6109 - auc: 0.6591 - loss: 0.7241 - precision: 0.6146 - recall: 0.6109 - val_acc: 0.6319 - val_auc: 0.6798 - val_loss: 0.6582 - val_precision: 0.6957 - val_recall: 0.5298
Epo

In [11]:
unfreeze_candidates = [2, 4, 6]
comparison_results = []
comparison_histories = {}
comparison_roc = {}

for num_blocks_to_unfreeze in unfreeze_candidates:
    print(f"\n===== Stage 2 experiment: unfreeze last {num_blocks_to_unfreeze} transformer blocks =====")

    stage2_model = keras.models.load_model(stage1_ckpt, compile=False)
    stage2_backbone = next(
        layer for layer in stage2_model.layers
        if isinstance(layer, keras_hub.models.ViTBackbone)
    )

    stage2_backbone.trainable = True
    all_backbone_layers, transformer_blocks = find_transformer_blocks(stage2_backbone)
    for layer in all_backbone_layers:
        layer.trainable = False

    unfrozen_blocks = transformer_blocks[-num_blocks_to_unfreeze:]
    for layer in unfrozen_blocks:
        layer.trainable = True

    patching_layer = stage2_backbone.get_layer("vit_patching_and_embedding")
    print("Found transformer blocks:", len(transformer_blocks))
    print("Unfrozen transformer blocks:", [layer.name for layer in unfrozen_blocks])
    print("Patching/embedding trainable:", patching_layer.trainable)
    print("Stage 2 trainable weights:", len(stage2_model.trainable_weights))

    compile_binary_model(stage2_model, learning_rate=1e-5, weight_decay=1e-4)

    ckpt_path = f"/mnt/e/SW_training_outputs/checkpoints/best_vit_unfreeze_{num_blocks_to_unfreeze}.keras"
    cb2 = [
        keras.callbacks.ModelCheckpoint(
            ckpt_path,
            monitor="val_auc",
            mode="max",
            save_best_only=True,
            verbose=1,
        ),
        keras.callbacks.EarlyStopping(
            monitor="val_auc",
            mode="max",
            patience=5,
            restore_best_weights=True,
            verbose=1,
        ),
    ]

    history2 = stage2_model.fit(
        train_ds_prep,
        validation_data=val_ds_prep,
        epochs=25,
        callbacks=cb2,
        verbose=1,
    )
    comparison_histories[num_blocks_to_unfreeze] = history2

    best_model = keras.models.load_model(ckpt_path, compile=False)
    compile_binary_model(best_model, learning_rate=1e-5, weight_decay=1e-4)
    test_metrics = best_model.evaluate(test_ds_prep, return_dict=True, verbose=1)
    test_true, test_prob = collect_predictions(test_ds_prep, best_model)
    test_pred = (test_prob >= 0.5).astype("int32")
    test_auc = roc_auc_score(test_true, test_prob)
    test_acc = accuracy_score(test_true, test_pred)
    test_fpr, test_tpr, _ = roc_curve(test_true, test_prob)

    comparison_roc[num_blocks_to_unfreeze] = {
        "fpr": test_fpr,
        "tpr": test_tpr,
        "auc": test_auc,
    }
    comparison_results.append(
        {
            "unfreeze_blocks": num_blocks_to_unfreeze,
            "ckpt_path": ckpt_path,
            "best_val_auc": float(max(history2.history["val_auc"])),
            "best_val_acc": float(max(history2.history["val_acc"])),
            "best_val_loss": float(min(history2.history["val_loss"])),
            "test_auc": float(test_auc),
            "test_acc": float(test_acc),
            "test_loss": float(test_metrics["loss"]),
            "test_precision": float(test_metrics["precision"]),
            "test_recall": float(test_metrics["recall"]),
        }
    )

comparison_results = sorted(comparison_results, key=lambda item: item["unfreeze_blocks"])
best_result = max(comparison_results, key=lambda item: item["test_auc"])
best_unfreeze = best_result["unfreeze_blocks"]
best_ckpt_path = best_result["ckpt_path"]
print("\nBest setting by test AUC:", best_result)



===== Stage 2 experiment: unfreeze last 2 transformer blocks =====
Found transformer blocks: 12
Unfrozen transformer blocks: ['tranformer_block_2', 'tranformer_block_1']
Patching/embedding trainable: False
Stage 2 trainable weights: 38
Epoch 1/25
142/142 ━━━━━━━━━━━━━━━━━━━━ 0s 273ms/step - acc: 0.6113 - auc: 0.6566 - loss: 0.7345 - precision: 0.6263 - recall: 0.6141
Epoch 1: val_auc improved from None to 0.70804, saving model to /mnt/e/SW_training_outputs/checkpoints/best_vit_unfreeze_2.keras

Epoch 1: finished saving model to /mnt/e/SW_training_outputs/checkpoints/best_vit_unfreeze_2.keras
142/142 ━━━━━━━━━━━━━━━━━━━━ 151s 890ms/step - acc: 0.6109 - auc: 0.6551 - loss: 0.7351 - precision: 0.6148 - recall: 0.6100 - val_acc: 0.6389 - val_auc: 0.7080 - val_loss: 0.6191 - val_precision: 0.6667 - val_recall: 0.5850
Epoch 2/25


2026-03-11 12:25:53.674795: I tensorflow/core/kernels/data/shuffle_dataset_op.cc:453] ShuffleDatasetV3:18: Filling up shuffle buffer (this may take a while): 52 of 1000
2026-03-11 12:26:03.697407: I tensorflow/core/kernels/data/shuffle_dataset_op.cc:453] ShuffleDatasetV3:18: Filling up shuffle buffer (this may take a while): 105 of 1000
2026-03-11 12:26:10.142399: I tensorflow/core/kernels/data/shuffle_dataset_op.cc:483] Shuffle buffer filled.
2026-03-11 12:26:10.225707: W tensorflow/core/kernels/data/prefetch_autotuner.cc:55] Prefetch autotuner tried to allocate 9634048 bytes after encountering the first element of size 9634048 bytes.This already causes the autotune ram budget to be exceeded. To stay within the ram budget, either increase the ram budget or reduce element size


142/142 ━━━━━━━━━━━━━━━━━━━━ 0s 267ms/step - acc: 0.6260 - auc: 0.6698 - loss: 0.7020 - precision: 0.6336 - recall: 0.6241
Epoch 2: val_auc did not improve from 0.70804
142/142 ━━━━━━━━━━━━━━━━━━━━ 72s 303ms/step - acc: 0.6069 - auc: 0.6477 - loss: 0.7223 - precision: 0.6114 - recall: 0.6039 - val_acc: 0.6146 - val_auc: 0.6870 - val_loss: 0.6317 - val_precision: 0.6092 - val_recall: 0.7114
Epoch 3/25


2026-03-11 12:26:54.603383: W tensorflow/core/kernels/data/prefetch_autotuner.cc:55] Prefetch autotuner tried to allocate 9634048 bytes after encountering the first element of size 9634048 bytes.This already causes the autotune ram budget to be exceeded. To stay within the ram budget, either increase the ram budget or reduce element size


142/142 ━━━━━━━━━━━━━━━━━━━━ 0s 265ms/step - acc: 0.5876 - auc: 0.6579 - loss: 0.7119 - precision: 0.5831 - recall: 0.6120
Epoch 3: val_auc improved from 0.70804 to 0.71939, saving model to /mnt/e/SW_training_outputs/checkpoints/best_vit_unfreeze_2.keras

Epoch 3: finished saving model to /mnt/e/SW_training_outputs/checkpoints/best_vit_unfreeze_2.keras
142/142 ━━━━━━━━━━━━━━━━━━━━ 70s 484ms/step - acc: 0.6056 - auc: 0.6622 - loss: 0.7052 - precision: 0.6063 - recall: 0.6196 - val_acc: 0.6319 - val_auc: 0.7194 - val_loss: 0.6163 - val_precision: 0.6581 - val_recall: 0.5385
Epoch 4/25
142/142 ━━━━━━━━━━━━━━━━━━━━ 0s 266ms/step - acc: 0.6201 - auc: 0.6840 - loss: 0.6822 - precision: 0.6256 - recall: 0.6238
Epoch 4: val_auc improved from 0.71939 to 0.72043, saving model to /mnt/e/SW_training_outputs/checkpoints/best_vit_unfreeze_2.keras

Epoch 4: finished saving model to /mnt/e/SW_training_outputs/checkpoints/best_vit_unfreeze_2.keras
142/142 ━━━━━━━━━━━━━━━━━━━━ 79s 547ms/step - acc: 0.62

...

142/142 ━━━━━━━━━━━━━━━━━━━━ 0s 264ms/step - acc: 0.6253 - auc: 0.6638 - loss: 0.7029 - precision: 0.6390 - recall: 0.6270
Epoch 6: val_auc did not improve from 0.74935
142/142 ━━━━━━━━━━━━━━━━━━━━ 70s 292ms/step - acc: 0.6316 - auc: 0.6783 - loss: 0.6868 - precision: 0.6352 - recall: 0.6319 - val_acc: 0.6667 - val_auc: 0.7418 - val_loss: 0.5929 - val_precision: 0.6774 - val_recall: 0.6954
Epoch 7/25
142/142 ━━━━━━━━━━━━━━━━━━━━ 0s 267ms/step - acc: 0.6545 - auc: 0.7094 - loss: 0.6486 - precision: 0.6575 - recall: 0.6696
Epoch 7: val_auc did not improve from 0.74935
142/142 ━━━━━━━━━━━━━━━━━━━━ 41s 281ms/step - acc: 0.6378 - auc: 0.6874 - loss: 0.6745 - precision: 0.6422 - recall: 0.6354 - val_acc: 0.6562 - val_auc: 0.7347 - val_loss: 0.6018 - val_precision: 0.6923 - val_recall: 0.6429
Epoch 8/25
142/142 ━━━━━━━━━━━━━━━━━━━━ 0s 268ms/step - acc: 0.6429 - auc: 0.6879 - loss: 0.6814 - precision: 0.6257 - recall: 0.6529
Epoch 8: val_auc did not improve from 0.74935
142/142 ━━━━━━━━━━━━━━━

2026-03-11 12:36:04.850443: E tensorflow/core/framework/node_def_util.cc:680] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),min=0; attr=output_types:list(type),min=1; attr=output_shapes:list(shape),min=1; attr=use_inter_op_parallelism:bool,default=true; attr=preserve_cardinality:bool,default=false; attr=force_synchronous:bool,default=false; attr=metadata:string,default=""> This may be expected if your graph generating binary is newer  than this binary. Unknown attributes will be ignored. NodeDef: {{node ParallelMapDatasetV2/_14}}



===== Stage 2 experiment: unfreeze last 4 transformer blocks =====
Found transformer blocks: 12
Unfrozen transformer blocks: ['tranformer_block_4', 'tranformer_block_3', 'tranformer_block_2', 'tranformer_block_1']
Patching/embedding trainable: False
Stage 2 trainable weights: 70
Epoch 1/25


2026-03-11 12:36:54.031318: I tensorflow/core/kernels/data/shuffle_dataset_op.cc:453] ShuffleDatasetV3:18: Filling up shuffle buffer (this may take a while): 53 of 1000
2026-03-11 12:37:10.644826: I tensorflow/core/kernels/data/shuffle_dataset_op.cc:483] Shuffle buffer filled.
2026-03-11 12:37:20.667613: W external/local_xla/xla/tsl/framework/bfc_allocator.cc:501] Allocator (GPU_0_bfc) ran out of memory trying to allocate 9.23MiB (rounded to 9682944)requested by op StatefulPartitionedCall/functional_2_1/vi_t_backbone_1/vit_encoder_1/tranformer_block_10_1/ln_2_1/mul_2
If the cause is memory fragmentation maybe the environment variable 'TF_GPU_ALLOCATOR=cuda_malloc_async' will improve the situation. 
Current allocation summary follows.
Current allocation summary follows.
2026-03-11 12:37:20.667728: I external/local_xla/xla/tsl/framework/bfc_allocator.cc:1049] BFCAllocator dump for GPU_0_bfc
2026-03-11 12:37:20.668153: I external/local_xla/xla/tsl/framework/bfc_allocator.cc:1056] Bin (256

ResourceExhaustedError: Graph execution error:

Detected at node functional_2_1/vi_t_backbone_1/vit_encoder_1/tranformer_block_10_1/ln_2_1/mul_2 defined at (most recent call last):
  File "<frozen runpy>", line 198, in _run_module_as_main

  File "<frozen runpy>", line 88, in _run_code

  File "/home/dell/miniconda3/envs/tf218/lib/python3.11/site-packages/ipykernel_launcher.py", line 18, in <module>

  File "/home/dell/miniconda3/envs/tf218/lib/python3.11/site-packages/traitlets/config/application.py", line 1075, in launch_instance

  File "/home/dell/miniconda3/envs/tf218/lib/python3.11/site-packages/ipykernel/kernelapp.py", line 758, in start

  File "/home/dell/miniconda3/envs/tf218/lib/python3.11/site-packages/tornado/platform/asyncio.py", line 211, in start

  File "/home/dell/miniconda3/envs/tf218/lib/python3.11/asyncio/base_events.py", line 608, in run_forever

  File "/home/dell/miniconda3/envs/tf218/lib/python3.11/asyncio/base_events.py", line 1936, in _run_once

  File "/home

In [ ]:
print("ViT unfreeze comparison summary:")
for result in comparison_results:
    print(
        f"unfreeze={result['unfreeze_blocks']:>2} | "
        f"best_val_loss={result['best_val_loss']:.4f} | "
        f"best_val_acc={result['best_val_acc']:.4f} | "
        f"best_val_auc={result['best_val_auc']:.4f} | "
        f"test_loss={result['test_loss']:.4f} | "
        f"test_acc={result['test_acc']:.4f} | "
        f"test_auc={result['test_auc']:.4f}"
    )


In [ ]:
block_values = [item["unfreeze_blocks"] for item in comparison_results]

fig, axes = plt.subplots(1, 3, figsize=(18, 4))
for depth in block_values:
    history = comparison_histories[depth].history
    axes[0].plot(history["loss"], label=f"Train {depth}", alpha=0.85)
    axes[0].plot(history["val_loss"], linestyle="--", label=f"Val {depth}", alpha=0.85)
    axes[1].plot(history["acc"], label=f"Train {depth}", alpha=0.85)
    axes[1].plot(history["val_acc"], linestyle="--", label=f"Val {depth}", alpha=0.85)
    axes[2].plot(history["auc"], label=f"Train {depth}", alpha=0.85)
    axes[2].plot(history["val_auc"], linestyle="--", label=f"Val {depth}", alpha=0.85)

axes[0].set_title("Loss Curves by Unfreeze Depth")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Loss")
axes[0].grid(True, alpha=0.3)

axes[1].set_title("Accuracy Curves by Unfreeze Depth")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Accuracy")
axes[1].grid(True, alpha=0.3)

axes[2].set_title("AUC Curves by Unfreeze Depth")
axes[2].set_xlabel("Epoch")
axes[2].set_ylabel("AUC")
axes[2].grid(True, alpha=0.3)

handles, labels = axes[2].get_legend_handles_labels()
fig.legend(handles, labels, loc="center left", bbox_to_anchor=(1.01, 0.5), frameon=False)
plt.tight_layout(rect=[0, 0, 0.88, 1])
plt.show()


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 4))
metric_specs = [
    ("best_val_loss", "Best Val Loss", "tab:red"),
    ("best_val_acc", "Best Val Accuracy", "tab:green"),
    ("best_val_auc", "Best Val AUC", "tab:blue"),
]
labels = [str(v) for v in block_values]

for ax, (metric_name, title, color) in zip(axes, metric_specs):
    values = [item[metric_name] for item in comparison_results]
    bars = ax.bar(labels, values, color=color, alpha=0.85)
    ax.set_title(title)
    ax.set_xlabel("Unfrozen transformer blocks")
    ax.grid(axis="y", alpha=0.3)
    for bar, value in zip(bars, values):
        ax.text(bar.get_x() + bar.get_width() / 2, value, f"{value:.3f}", ha="center", va="bottom", fontsize=9)

axes[0].set_ylabel("Score")
plt.tight_layout()
plt.show()


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 4))
test_metric_specs = [
    ("test_loss", "Test Loss", "tab:red"),
    ("test_acc", "Test Accuracy", "tab:green"),
    ("test_auc", "Test AUC", "tab:blue"),
]
labels = [str(v) for v in block_values]

for ax, (metric_name, title, color) in zip(axes, test_metric_specs):
    values = [item[metric_name] for item in comparison_results]
    bars = ax.bar(labels, values, color=color, alpha=0.85)
    ax.set_title(title)
    ax.set_xlabel("Unfrozen transformer blocks")
    ax.grid(axis="y", alpha=0.3)
    for bar, value in zip(bars, values):
        ax.text(bar.get_x() + bar.get_width() / 2, value, f"{value:.3f}", ha="center", va="bottom", fontsize=9)

axes[0].set_ylabel("Score")
plt.tight_layout()
plt.show()


In [ ]:
plt.figure(figsize=(6, 6))
for depth in block_values:
    roc_info = comparison_roc[depth]
    plt.plot(
        roc_info["fpr"],
        roc_info["tpr"],
        label=f"Unfreeze {depth} (AUC = {roc_info['auc']:.4f})",
    )

plt.plot([0, 1], [0, 1], linestyle="--", color="gray")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("Test ROC Comparison")
plt.legend(loc="lower right")
plt.grid(True, alpha=0.3)
plt.show()


In [ ]:
best_model = keras.models.load_model(best_ckpt_path, compile=False)
compile_binary_model(best_model, learning_rate=1e-5, weight_decay=1e-4)
print("Loaded best comparison model from:", best_ckpt_path)
print("Best unfreeze depth:", best_unfreeze)

test_metrics = best_model.evaluate(test_ds_prep, return_dict=True)
print("Evaluation metrics:", test_metrics)

test_true, test_prob = collect_predictions(test_ds_prep, best_model)
test_pred = (test_prob >= 0.5).astype("int32")
test_fpr, test_tpr, _ = roc_curve(test_true, test_prob)
test_auc = roc_auc_score(test_true, test_prob)
test_acc = accuracy_score(test_true, test_pred)
cm = confusion_matrix(test_true, test_pred)

print(f"Test AUC: {test_auc:.4f}")
print(f"Test Accuracy: {test_acc:.4f}")
print("Confusion Matrix:\n", cm)
print("Classification Report:\n", classification_report(test_true, test_pred, target_names=class_names))

plt.figure(figsize=(6, 6))
plt.plot(test_fpr, test_tpr, label=f"ROC curve (AUC = {test_auc:.4f})")
plt.plot([0, 1], [0, 1], linestyle="--", color="gray")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title(f"ROC Curve (Best Depth = {best_unfreeze})")
plt.legend(loc="lower right")
plt.grid(True, alpha=0.3)
plt.show()
